<a href="https://colab.research.google.com/github/Natthakris/Design-Thinking-2025/blob/main/Self-carring-Web-scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import re

# URL ของเว็บไซต์ที่ต้องการดึงข้อมูล
base_url = "https://multimedia.anamai.moph.go.th/help-knowledgs/"
all_scraped_content_strings = []
page_num = 1
max_pages_to_scrape = 20 # กำหนดจำนวนหน้าสูงสุดที่ต้องการดึงข้อมูล

while page_num <= max_pages_to_scrape:
    # สร้าง URL สำหรับหน้าปัจจุบัน
    current_page_url = f"{base_url}page/{page_num}/" if page_num > 1 else base_url
    print(f"กำลังดึงข้อมูลจาก URL: {current_page_url}")

    try:
        page = requests.get(current_page_url)
        page.raise_for_status() # ตรวจสอบข้อผิดพลาด HTTP
    except requests.exceptions.RequestException as e:
        print(f"เกิดข้อผิดพลาดในการดึงหน้า {current_page_url}: {e}")
        break # หยุดการทำงานหากมีข้อผิดพลาดในการร้องขอ

    soup = BeautifulSoup(page.content, "html.parser")
    current_page_content_elements = soup.find_all(class_="col-12 col-md-4 col-lg-4")

    if not current_page_content_elements:
        print(f"ไม่พบเนื้อหาหรือถึงหน้าสุดท้ายแล้วบน {current_page_url}. หยุดการดึงข้อมูล.")
        break # ไม่มีเนื้อหาเพิ่มเติม, หยุดลูป

    # แปลงรายการ Tag object ให้เป็นสตริงแล้วเพิ่มเข้าไปใน all_scraped_content_strings
    all_scraped_content_strings.append("".join([str(element) for element in current_page_content_elements]))
    page_num += 1

# รวมเนื้อหาทั้งหมดที่ดึงมาได้จากทุกหน้าเข้าเป็นสตริงเดียวสำหรับการดำเนินการ regex ในเซลล์ถัดไป
content = "".join(all_scraped_content_strings)
print(f"ดึงข้อมูลเสร็จสิ้นจาก {page_num - 1} หน้า")

กำลังดึงข้อมูลจาก URL: https://multimedia.anamai.moph.go.th/help-knowledgs/
กำลังดึงข้อมูลจาก URL: https://multimedia.anamai.moph.go.th/help-knowledgs/page/2/
กำลังดึงข้อมูลจาก URL: https://multimedia.anamai.moph.go.th/help-knowledgs/page/3/
กำลังดึงข้อมูลจาก URL: https://multimedia.anamai.moph.go.th/help-knowledgs/page/4/
กำลังดึงข้อมูลจาก URL: https://multimedia.anamai.moph.go.th/help-knowledgs/page/5/
กำลังดึงข้อมูลจาก URL: https://multimedia.anamai.moph.go.th/help-knowledgs/page/6/
กำลังดึงข้อมูลจาก URL: https://multimedia.anamai.moph.go.th/help-knowledgs/page/7/
กำลังดึงข้อมูลจาก URL: https://multimedia.anamai.moph.go.th/help-knowledgs/page/8/
กำลังดึงข้อมูลจาก URL: https://multimedia.anamai.moph.go.th/help-knowledgs/page/9/
กำลังดึงข้อมูลจาก URL: https://multimedia.anamai.moph.go.th/help-knowledgs/page/10/
กำลังดึงข้อมูลจาก URL: https://multimedia.anamai.moph.go.th/help-knowledgs/page/11/
กำลังดึงข้อมูลจาก URL: https://multimedia.anamai.moph.go.th/help-knowledgs/page/12/
กำลังดึง

In [2]:
# เลือกข้อมูลที่ต้องการสกัด
re_titles = r'title="(.*?)">'
titles_list = re.findall(re_titles, content)

re_descriptions = r'<p>(.*?)</p>'
descriptions_list = re.findall(re_descriptions, content)

re_links = r'href="(.*?)"'
links_list = re.findall(re_links, content)

In [5]:
import pandas as pd

# ตรวจสอบและปรับความยาวของ lists ให้เท่ากันก่อนสร้าง DataFrame
# จากสถานะของ kernel, titles_list และ descriptions_list มีความยาว 50
# ในขณะที่ links_list มีความยาว 100
# เพื่อแก้ไข ValueError: All arrays must be of the same length, เราจะตัด links_list ให้เหลือ 50 รายการ
# ควรพิจารณาปรับปรุงตรรกะการสกัดข้อมูลในเซลล์ก่อนหน้าเพื่อให้ได้ข้อมูลที่จับคู่กันอย่างถูกต้องตั้งแต่แรก
min_len = min(len(titles_list), len(descriptions_list), len(links_list))

titles_list = titles_list[:min_len]
descriptions_list = descriptions_list[:min_len]
links_list = links_list[:min_len]

# สร้าง DataFrame จาก lists ของ titles และ descriptions
data = {'Title': titles_list, 'Description': descriptions_list, 'Link': links_list}
df_output = pd.DataFrame(data)

# บันทึก DataFrame ลงในไฟล์ .csv
# ใช้ encoding='utf-8-sig' เพื่อรองรับภาษาไทยได้ดีขึ้นและให้ Excel อ่านได้โดยไม่มีปัญหา
# index=False เพื่อไม่บันทึกคอลัมน์ index ของ DataFrame ลงในไฟล์ CSV
# sep=',' กำหนดตัวคั่นเป็น comma ซึ่งเป็นมาตรฐานสำหรับ CSV
df_output.to_csv('output.csv', index=False, encoding='utf-8-sig', sep=',')

print("บันทึกข้อมูลลงในไฟล์ 'output.csv' เรียบร้อยแล้วด้วย pandas")

บันทึกข้อมูลลงในไฟล์ 'output.csv' เรียบร้อยแล้วด้วย pandas
